# RQ1 — Notebook 2: LSE Computation

**Research Question**: Which clustering method generates the best pseudo-labels for downstream classification?

This notebook computes Label Substitution Efficiency (LSE) for the 113-dataset manifest. The current run collected 107 dataset-level results, dropped 13 fully failed or unusable datasets, and saved **94 usable rows** to the meta-training table.

**LSE = balanced_accuracy(RF on pseudo-labels) / balanced_accuracy(RF on true labels)**

Balanced accuracy is used throughout for imbalanced datasets, and the fixed RF uses `class_weight='balanced'`.

**Feature preprocessing**:
- Numeric features: median imputation fit on train only, then `StandardScaler`
- Categorical features: one-hot encoded via `pd.get_dummies` (`dummy_na=False`)
- Clustering uses scaled train features; RF evaluation uses the imputed feature matrix

**Current output highlights**:
- `data/meta_table/meta_training.csv` shape: `(94, 9)`
- Best-method distribution: k-means 28, GMM 21, agglomerative 17, autoencoder 13, dictionary learning 10, DBSCAN 5
- Mean LSE: GMM 0.664, agglomerative 0.644, k-means 0.640, autoencoder 0.623, dictionary learning 0.559, DBSCAN 0.521
- Mean per-dataset LSE standard deviation: 0.107, indicating usable method-separation signal

> Fresh-start note: delete `data/meta_table/lse_checkpoint.csv` if you changed the LSE formula, feature preprocessing, method implementations, or manifest.


In [1]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import openml
import torch

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

ROOT       = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR    = os.path.join(ROOT, 'data', 'raw')
META_DIR   = os.path.join(ROOT, 'data', 'meta_table')
MANIFEST   = os.path.join(META_DIR, 'dataset_manifest.csv')
CHECKPOINT = os.path.join(META_DIR, 'lse_checkpoint.csv')
OUTPUT     = os.path.join(META_DIR, 'meta_training.csv')

sys.path.insert(0, os.path.join(ROOT, 'src'))
openml.config.cache_directory = RAW_DIR

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print('Paths OK')

Paths OK


In [2]:
SHOWCASE_IDS = {61, 187, 15, 53, 40966, 37, 54, 1590, 1597}

manifest = pd.read_csv(MANIFEST)
leaked = set(manifest['dataset_id']) & SHOWCASE_IDS
assert len(leaked) == 0, f'Showcase leak in manifest: {leaked}'

print(f'Manifest: {len(manifest)} datasets')
manifest.head()

Manifest: 113 datasets


,dataset_id,openml_did,name,n_instances,n_features,n_classes,n_numerical_features,n_categorical_features,missing_value_fraction,class_balance_ratio,openml_version,download_timestamp,class_count_bin,size_bin,balance_bin,stratum
0,46850,46850,hepatitis_c_virus_hcv_for_egyptian_patients,1385,28,4,28,0,0.0,0.9171,2,2026-05-02T11:56:42Z,3-4,medium,balanced,3-4_medium_balanced
1,41881,41881,FOREX_chfjpy-hour-High,43825,10,2,10,1,0.0,0.9176,1,2026-05-02T11:56:42Z,binary,large,balanced,binary_large_balanced
2,44507,44507,steel-plates-fault_seed_4_nrows_2000_nclasses_...,1941,27,7,27,0,0.0,0.0817,1,2026-05-02T11:56:42Z,5-7,medium,imbalanced,5-7_medium_imbalanced
3,44227,44227,South_Asian_Churn_dataset,2000,9,2,9,4,0.0,1.0000,1,2026-05-02T11:56:42Z,binary,medium,balanced,binary_medium_balanced
4,44624,44624,mfeat-factors_seed_1_nrows_2000_nclasses_10_nc...,2000,100,10,100,0,0.0,1.0000,1,2026-05-02T11:56:42Z,8-10,medium,balanced,8-10_medium_balanced


In [3]:
def load_and_split(dataset_id):
    ds = openml.datasets.get_dataset(
        dataset_id, download_data=True,
        download_qualities=False,
        download_features_meta_data=False,
    )
    X, y, _, _ = ds.get_data(
        dataset_format='dataframe',
        target=ds.default_target_attribute,
    )

    # Numeric features kept as-is; categorical features one-hot encoded.
    # get_dummies with dummy_na=False: rows with a missing categorical value
    # receive all-zeros for that feature's columns (implicit zero imputation).
    X_num = X.select_dtypes(include=[np.number])
    X_cat = X.select_dtypes(exclude=[np.number])

    if X_cat.shape[1] > 0:
        X_cat_enc = pd.get_dummies(X_cat, dummy_na=False).astype(float)
        X_out = pd.concat([X_num.reset_index(drop=True),
                           X_cat_enc.reset_index(drop=True)], axis=1)
    else:
        X_out = X_num

    le = LabelEncoder()
    y_enc = le.fit_transform(y.astype(str))

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_out.values.astype(float), y_enc, test_size=0.2,
        random_state=SEED, stratify=y_enc,
    )
    return X_tr, X_te, y_tr, y_te


def impute(X_tr, X_te):
    """Median imputation fit on training set only; handles numeric NaN."""
    imp = SimpleImputer(strategy='median')
    return imp.fit_transform(X_tr), imp.transform(X_te)


def scale(X_tr, X_te):
    sc = StandardScaler()
    return sc.fit_transform(X_tr), sc.transform(X_te)

In [4]:
from lse import compute_lse, groundtruth_accuracy
from clustering import (
    pseudo_kmeans, pseudo_dbscan, pseudo_agglomerative,
    pseudo_gmm, pseudo_autoencoder, pseudo_dictlearn,
)
print('Modules loaded')

Modules loaded


In [5]:
METHODS = {
    'LSE_kmeans'   : pseudo_kmeans,
    'LSE_dbscan'   : pseudo_dbscan,
    'LSE_agg'      : pseudo_agglomerative,
    'LSE_gmm'      : pseudo_gmm,
    'LSE_autoenc'  : pseudo_autoencoder,
    'LSE_dictlearn': pseudo_dictlearn,
}

# Minimum balanced ground-truth RF accuracy — datasets below this threshold
# have label structure too weak for pseudo-labeling to be meaningful.
MIN_GT_ACC = 0.30

if os.path.exists(CHECKPOINT):
    done_df  = pd.read_csv(CHECKPOINT)
    done_ids = set(done_df['dataset_id'])
    results  = done_df.to_dict('records')
    print(f'Resuming — {len(done_ids)} datasets already processed')
else:
    done_ids = set()
    results  = []
    print('Starting fresh')

all_diagnostics = []
total = len(manifest)

for i, row in manifest.iterrows():
    did   = int(row['dataset_id'])
    name  = row['name']
    n_cls = int(row['n_classes'])

    if did in done_ids:
        continue

    t0  = time.time()
    rec = {'dataset_id': did}

    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)

        if X_tr.shape[1] == 0:
            raise ValueError('No features after encoding')

        X_tr, X_te     = impute(X_tr, X_te)
        X_tr_sc, X_te_sc = scale(X_tr, X_te)
        gt_acc = groundtruth_accuracy(X_tr, y_tr, X_te, y_te)

        if gt_acc < MIN_GT_ACC:
            print(f'[{i+1:3d}/{total}] SKIP  id={did}  (balanced_gt={gt_acc:.3f} < {MIN_GT_ACC})')
            done_ids.add(did)
            continue

        print(f'[{i+1:3d}/{total}] {name[:35]:35s}  balanced_gt={gt_acc:.3f}  n_cls={n_cls}  d={X_tr.shape[1]}')

        for col, fn in METHODS.items():
            rec[col] = float('nan')
            try:
                pseudo = fn(X_tr_sc, n_cls)
                lse, diag = compute_lse(
                    X_tr, y_tr, X_te, y_te, pseudo, gt_acc,
                    method_name=col.replace('LSE_', ''),
                    dataset_name=name, verbose=True,
                )
                rec[col] = round(lse, 4)
                all_diagnostics.append(diag)
            except Exception as e:
                print(f'    {col} FAILED: {e}')
                all_diagnostics.append({'dataset': name,
                                         'method': col.replace('LSE_', ''),
                                         'failed': True, 'error': str(e)[:200]})

        elapsed = time.time() - t0
        valid_cols = [c for c in METHODS if not np.isnan(rec.get(c, np.nan))]
        best = max(valid_cols, key=lambda c: rec.get(c, -1)) if valid_cols else 'FAILED'
        rec['best_method'] = best.replace('LSE_', '')
        rec['gt_accuracy'] = round(gt_acc, 4)

        results.append(rec)
        done_ids.add(did)
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False)
        print(f'           -> best={rec["best_method"]}  ({elapsed:.1f}s)')

    except Exception as e:
        print(f'[{i+1:3d}/{total}] FAIL  id={did}  {name}  — {e}')
        rec.update({c: float('nan') for c in METHODS})
        rec['best_method'] = 'FAILED'
        rec['gt_accuracy']  = float('nan')
        results.append(rec)
        done_ids.add(did)
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False)

print(f'\nDone. {len(results)} rows collected.')

Starting fresh
[  1/113] SKIP  id=46850  (balanced_gt=0.225 < 0.3)
[  2/113] FAIL  id=41881  FOREX_chfjpy-hour-High  — Cannot cast DatetimeArray to dtype float64
[  3/113] steel-plates-fault_seed_4_nrows_200  balanced_gt=0.788  n_cls=7  d=27
    kmeans      n_cl=7/7  bal_acc=0.399  gt_bal=0.788  lse=0.506  lift=0.397
    dbscan      n_cl=4/7  bal_acc=0.238  gt_bal=0.788  lse=0.302  lift=0.148  [CLUSTER_COLLAPSE, MAPPING_COLLAPSE]
    agg         n_cl=7/7  bal_acc=0.377  gt_bal=0.788  lse=0.479  lift=0.363
    gmm         n_cl=7/7  bal_acc=0.417  gt_bal=0.788  lse=0.529  lift=0.425
    autoenc     n_cl=7/7  bal_acc=0.259  gt_bal=0.788  lse=0.329  lift=0.181
    dictlearn   n_cl=7/7  bal_acc=0.335  gt_bal=0.788  lse=0.425  lift=0.298
           -> best=gmm  (41.0s)
[  4/113] South_Asian_Churn_dataset            balanced_gt=0.755  n_cls=2  d=28
    kmeans      n_cl=2/2  bal_acc=0.500  gt_bal=0.755  lse=0.662  lift=0.000  [MATCHES_MAJORITY]
    dbscan      n_cl=7/2  bal_acc=0.535  gt_bal=0

In [6]:
df = pd.DataFrame(results)
lse_cols = list(METHODS.keys())

# Drop fully-failed datasets
all_nan = df[lse_cols].isna().all(axis=1)
if all_nan.any():
    print(f'Dropping {all_nan.sum()} fully-failed datasets')
    df = df[~all_nan].reset_index(drop=True)

# Recompute best_method robustly
def _best(row):
    vals = {c: row[c] for c in lse_cols if not np.isnan(row[c])}
    return max(vals, key=vals.get).replace('LSE_', '') if vals else 'FAILED'

df['best_method'] = df.apply(_best, axis=1)
col_order = ['dataset_id'] + lse_cols + ['best_method', 'gt_accuracy']
df = df[col_order]

leaked = set(df['dataset_id']) & SHOWCASE_IDS
assert len(leaked) == 0, f'Showcase leak: {leaked}'

df.to_csv(OUTPUT, index=False)
print(f'Saved → {OUTPUT}  shape={df.shape}')

Dropping 13 fully-failed datasets
Saved → c:\MLResearch\data\meta_table\meta_training.csv  shape=(94, 9)


In [7]:
if all_diagnostics:
    diag_df = pd.DataFrame(all_diagnostics)
    DIAG_PATH = os.path.join(META_DIR, 'diagnostics.csv')
    diag_df.to_csv(DIAG_PATH, index=False)

    print('=== Failure-mode frequency ===')
    for flag in ['cluster_collapse', 'cluster_degenerate', 'mapping_collapse',
                 'matches_majority', 'rf_underfit_pseudo']:
        if flag in diag_df.columns:
            pct = diag_df[flag].fillna(False).mean() * 100
            print(f'  {flag:25s}  {pct:5.1f}%')
else:
    print('No diagnostics (all datasets loaded from checkpoint). Delete checkpoint to regenerate.')

=== Failure-mode frequency ===
  cluster_collapse             6.9%
  cluster_degenerate          15.2%
  mapping_collapse             6.9%
  matches_majority            14.7%
  rf_underfit_pseudo           0.0%


In [10]:
df = pd.read_csv(OUTPUT)

print('=== LSE descriptive statistics (balanced accuracy ratio) ===')
print(df[lse_cols].describe().round(3).to_string())

print('\n=== Best-method distribution ===')
print(df['best_method'].value_counts())

print('\n=== NaN counts per method ===')
print(df[lse_cols].isna().sum())

print('\n=== Per-dataset LSE std across methods (meta-learnability signal) ===')
lse_std = df[lse_cols].std(axis=1)
print(lse_std.describe().round(3).to_string())
print('(Mean std < 0.05 → methods too similar for meta-learner to distinguish)')

for col in lse_cols:
    valid = df[col].dropna()
    assert (valid >= 0).all(), f'{col} has negative LSE'
    outliers = valid[valid > 1.0]
    if len(outliers):
        print(f'  NOTE: {col} has {len(outliers)} value(s) > 1.0 '
            f'(max={outliers.max():.4f}) — plausible evaluation noise, kept as-is')
    assert (valid <= 2.0).all(), f'{col} has LSE > 2.0, likely a computation error'

print('\nAll sanity checks passed.')
print('Ready for 03_method_redundancy.ipynb')

=== LSE descriptive statistics (balanced accuracy ratio) ===
       LSE_kmeans  LSE_dbscan  LSE_agg  LSE_gmm  LSE_autoenc  LSE_dictlearn
count      94.000      92.000   84.000   94.000       94.000         92.000
mean        0.640       0.521    0.644    0.664        0.623          0.559
std         0.226       0.254    0.226    0.243        0.231          0.188
min         0.235       0.106    0.229    0.270        0.257          0.227
25%         0.502       0.332    0.510    0.520        0.460          0.431
50%         0.610       0.496    0.601    0.619        0.599          0.506
75%         0.792       0.665    0.822    0.822        0.722          0.696
max         1.333       1.191    1.488    1.678        1.700          1.096

=== Best-method distribution ===
best_method
kmeans       28
gmm          21
agg          17
autoenc      13
dictlearn    10
dbscan        5
Name: count, dtype: int64

=== NaN counts per method ===
LSE_kmeans        0
LSE_dbscan        2
LSE_agg         